## Executive Summary

**Purpose:** Size the parts that carry the gear forces into the housing: the joint between each gear and its shaft, the two shafts, and the four bearings, including the **axial force** that a helical mesh adds.

**What it does:** `size_pair_components` takes a `HelicalPair` and its rating, together with the shaft diameters and layout choices (`PairComponentInputs`), and returns `PairComponents`. That holds the hub joint of pinion and gear (key, press fit, adhesive, or teeth cut on the shaft), the bearing reactions, the equivalent load, the required and selected deep groove ball bearing for every bearing, the minimum shaft diameters, the shaft deflection at the mesh, and the housing bore tolerances. It also lists every check with its value and limit.

**Units:** mm, N, N·m, MPa, h.

## How It Fits Into the Helical Gear Workflow

**Upstream:** `gears.helical.layout` (`HelicalPair.forces`), `gears.helical.rating` (torques, pitch line velocity), `gears.components` (parallel keys, press fits, adhesive joints, hob run-out, lubrication), `gears.manufacturing` (ISO 286 limits, ISO 1328 helix tolerance).

**Downstream:** `gears.helical.design` (search, report and parts list).

In [ ]:
#| default_exp gears.helical.components

In [ ]:
#| export
from __future__ import annotations

import math
from dataclasses import dataclass
from types import SimpleNamespace

from emachines.gears.components import (ParallelKey, Check, select_press_fit, press_fit, adhesive_torque,
                                        hob_runout, lubrication_advice)
from emachines.gears.manufacturing import it_grade, iso1328_tolerances
from emachines.gears.helical.geometry import HelicalGear
from emachines.gears.helical.layout import HelicalPair
from emachines.gears.helical.rating import HelicalPairRating

## Inputs

The arrangement is the usual one for a single stage: each shaft runs in **two bearings, A and B**, with the gear between them. The gear sits at `gear_position` of the span from bearing A. Bearing A is the **located** bearing. It takes the axial force in both directions, while B floats axially. The spans default to the face width plus one bearing width plus 2 × 3 mm of shoulder and clearance, which is about the shortest a housing allows.

`hub_joint1 = 'auto'` cuts the pinion teeth on its shaft when a bore would leave less than 1.2 tooth depths of rim under the roots. That is the usual outcome for a pinion on a 10–25 mm shaft at small modules.

In [ ]:
#| export
@dataclass(frozen=True)
class PairComponentInputs:
    """
    Shafts, joints and layout of a single-stage helical pair. Lengths in mm, stresses in MPa.

    shaft_d1, shaft_d2  : pinion (input) and gear (output) shaft diameter at the gear seat
    hub_joint1, hub_joint2 : 'key', 'press', 'adhesive', 'integral' (teeth cut on the shaft) or 'auto'
                          ('auto': integral if a bore leaves less than `rim_min` tooth depths, else key)
    press_fit           : fit for 'press' joints (None: lightest that passes)
    span1, span2        : bearing spans [mm] (None: estimated, see above)
    gear_position       : gear centre as a fraction of the span from bearing A
    bearing_life_h      : required L10h (None: the gear life)
    rim_min             : smallest rim under the roots for a bored gear, in tooth depths
    slip_safety         : torque and axial capacity of press fit or adhesive over K_A·loads
    sigma_shaft_allow   : allowable equivalent (von Mises) shaft stress at the gear seat, fatigue with notch [MPa]
    tau_shaft_allow     : allowable torsion stress in a pinion shaft at its root diameter [MPa]
    p_key_allow, sigma_hub_allow, tau_adhesive_allow, mu_fit, Rz_fit, hob_d : as in `gears.components`
    wall_min            : housing material left between the outer rings of pinion and gear bearings [mm]
    """
    shaft_d1: float
    shaft_d2: float
    hub_joint1: str = "auto"
    hub_joint2: str = "key"
    press_fit: str | None = None
    span1: float | None = None
    span2: float | None = None
    gear_position: float = 0.5
    bearing_life_h: float | None = None
    rim_min: float = 1.2
    slip_safety: float = 1.5
    sigma_shaft_allow: float = 80.0
    tau_shaft_allow: float = 60.0
    p_key_allow: float = 150.0
    sigma_hub_allow: float = 500.0
    tau_adhesive_allow: float = 8.0
    mu_fit: float = 0.12
    Rz_fit: float = 2.0
    hob_d: float = 40.0
    wall_min: float = 3.0

## Hub Joints and the Axial Force

The joint carries the torque $T$ and, for a helical gear, the axial force $F_a$. A **key** carries torque only. When the face is narrow, the hub is lengthened to at most $1.2\,d$ to get a longer key. The axial force needs a shaft shoulder on one side and a retaining ring, nut or spacer on the other. A **press fit** or an **adhesive** joint carries both through friction or shear. The force it must hold at its surface is the vector sum

$$F_{req} = S\,K_A\sqrt{\left(\frac{2000\,T}{d}\right)^2 + F_a^2} ,$$

against $\mu\,p_{min}\,\pi d L$ for a press fit (Lamé, `gears.components.press_fit`) and $\tau\,\pi d L$ for an adhesive. A pinion cut **on its shaft** has no joint. The checks are the torsion stress at its root diameter and the hob run-out to the next shoulder.

The rim under the tooth roots, $s_R = (d_f - d_{bore})/2 - t_2$, sets whether a bore is possible (`rim_min` tooth depths) and enters the rating through $Y_B$.

In [ ]:
#| export
def _resolve_joint(g: HelicalGear, joint: str, d_shaft: float, rim_min: float) -> str:
    if joint != "auto":
        return joint
    t2 = ParallelKey.for_shaft(d_shaft).t2 if d_shaft > 6 else 0.0
    rim = (g.df - d_shaft) / 2 - t2
    return "key" if rim >= rim_min * g.h else "integral"


def pair_rims(pair: HelicalPair, ci: PairComponentInputs) -> dict[str, float | None]:
    "Rim under the tooth roots [mm] of pinion and gear for the rim factor Y_B (None: teeth on the shaft)."
    out: dict[str, float | None] = {}
    for key, g, joint, d in (("pinion", pair.pinion, ci.hub_joint1, ci.shaft_d1),
                             ("gear", pair.gear, ci.hub_joint2, ci.shaft_d2)):
        j = _resolve_joint(g, joint, d, ci.rim_min)
        if j == "integral":
            out[key] = None
        else:
            t2 = ParallelKey.for_shaft(d).t2 if j == "key" else 0.0
            out[key] = (g.df - d) / 2 - t2
    return out


def size_hub(g: HelicalGear, joint: str, d: float, T: float, F_a: float, K_A: float,
             ci: PairComponentInputs) -> tuple[dict, list[Check]]:
    "Size the joint of gear g on a shaft of diameter d for torque T [N·m] and axial force F_a [N]."
    joint = _resolve_joint(g, joint, d, ci.rim_min)
    checks: list[Check] = []
    L = g.b
    hub: dict = {"joint": joint, "shaft_d": d}
    if joint != "integral":
        rim = (g.df - d) / 2 - (ParallelKey.for_shaft(d).t2 if joint == "key" else 0.0)
        hub["rim"] = rim
        checks.append(Check(f"{g.name} rim under the roots / tooth depth", rim / g.h, ci.rim_min, "", upper=False))
    F_req = ci.slip_safety * K_A * math.hypot(2000 * T / d, F_a)
    if joint == "key":
        key = ParallelKey.for_shaft(d)
        L_hub = max(L, math.ceil(1.2 * d))                # hub boss up to 1.2 d long when the face is narrow
        L_key = ParallelKey.standard_length(L_hub)
        p_key = key.pressure(K_A * T, L_key)
        hub.update(bore_fit="H7 (shaft k6)", key=f"DIN 6885-1 A {key.b:g} × {key.h:g} × {L_key:g}", key_b=key.b,
                   t1=key.t1, t2=key.t2, key_length=L_key, key_pressure=p_key, hub_length=L_hub,
                   axial=f"shaft shoulder + DIN 471 retaining ring or nut for F_a = {F_a:.0f} N (both directions)")
        checks.append(Check(f"{g.name} key flank pressure", p_key, ci.p_key_allow, "MPa"))
    elif joint == "press":
        T_eq = F_req * d / 2000                    # torque that needs the same friction force
        pf = (press_fit(d, g.df, L, ci.press_fit, ci.mu_fit, ci.Rz_fit) if ci.press_fit
              else select_press_fit(d, g.df, L, T_eq, ci.sigma_hub_allow, ci.mu_fit, ci.Rz_fit))
        F_cap = ci.mu_fit * pf.p_min * math.pi * d * L
        hub.update(bore_fit=pf.fit, press_fit=pf.fit, p_min=pf.p_min, p_max=pf.p_max, F_friction=F_cap,
                   sigma_hub=pf.sigma_hub, shrink_temperature=pf.T_join,
                   axial="friction carries F_a; add a shoulder for assembly location")
        checks.append(Check(f"{g.name} press fit friction / required (torque + axial)", F_cap / F_req, 1.0, "", upper=False))
        checks.append(Check(f"{g.name} press fit stress at the bore", pf.sigma_hub, ci.sigma_hub_allow, "MPa"))
    elif joint == "adhesive":
        F_cap = adhesive_torque(d, L, ci.tau_adhesive_allow) * 2000 / d
        hub.update(bore_fit="H7 (shaft k6) with anaerobic retaining compound", F_shear=F_cap,
                   axial="shoulder on the thrust side recommended")
        checks.append(Check(f"{g.name} adhesive shear capacity / required", F_cap / F_req, 1.0, "", upper=False))
    elif joint == "integral":
        tau = 16 * 1000 * K_A * T / (math.pi * g.df ** 3)
        L_ro = hob_runout(d / 2 - (g.df / 2 - 0.25 * g.mn), ci.hob_d)
        hub.update(bore_fit="none: teeth cut on the shaft", shaft_torsion=tau, hob_runout=L_ro,
                   note=("journals below the root: the hob runs out freely" if L_ro == 0 else
                         f"journal above the root: leave {L_ro:.1f} mm hob run-out to the shoulder"),
                   axial="carried by the shaft into the located bearing")
        checks.append(Check(f"{g.name} shaft torsion at the root diameter", tau, ci.tau_shaft_allow, "MPa"))
    else:
        raise ValueError("hub joint must be 'key', 'press', 'adhesive', 'integral' or 'auto'")
    return hub, checks

## Bearing Reactions

Put the gear at distance $l_A$ from bearing A and $l_B = l - l_A$ from bearing B. The mesh forces act at the operating pitch radius $r_w$. The tangential force $F_t$ and the separating force $F_r$ act at the gear, and the axial force $F_a$ at radius $r_w$ adds a moment $F_a r_w$ in the plane of $F_r$:

$$R_{A,t} = F_t\frac{l_B}{l},\quad R_{B,t} = F_t\frac{l_A}{l},\qquad
R_{A,r} = \frac{F_r l_B + F_a r_w}{l},\quad R_{B,r} = \frac{F_r l_A + F_a r_w}{l} .$$

The sign of the $F_a r_w$ term depends on the hand and the torque direction. Both bearings get the unfavourable sign here, because a motor drive runs both ways. The radial load of each bearing is $F_R = \sqrt{R_t^2 + R_r^2}$, and the located bearing A also takes $F_a$.

**Equivalent load, deep groove ball bearing** (ISO 281, typical for $f_0F_a/C_0 \approx 0.5$–1):

$$P = \begin{cases}F_R & F_a/F_R \le e = 0.3\\ 0.56\,F_R + 1.5\,F_a & F_a/F_R > e\end{cases}\qquad
C_{req} = P\left(\frac{60\,n\,L_{10h}}{10^6}\right)^{1/3} .$$

Loads are nominal (no $K_A$), as for a bearing life at the rated torque. A bearing bore larger than the shaft diameter means a stepped journal, which the component table lists.

**Housing wall:** the bearings of both shafts sit in the same wall on each side. Their outer rings must clear each other with some material between them: $(D_1 + D_2)/2 + $ `wall_min` $\le a_w$. At small centre distances this, not the gears, often sets the size of the stage. The bearing is picked from `DEEP_GROOVE_BEARINGS`, a short list of typical 6xx/60xx/62xx catalogue values. It is the smallest outside diameter with a bore at least the journal diameter and $C_r \ge C_{req}$. Confirm the part against the maker's catalogue, and check its $C_0$ against the peak load. When the axial force is a large share of the load, an angular contact pair or a four-point bearing at A is the better choice.

**Shaft** at the gear: bending moment $M = \sqrt{(R_{A,t}l_A)^2 + (\max(R_{A,r}l_A,\ R_{B,r}l_B))^2}$ and torque $T$. The minimum diameter is

$$d_{min} = \left(\frac{32\,K_A\sqrt{M^2 + 0.75\,T^2}}{\pi\,\sigma_{allow}}\right)^{1/3},$$

with $\sigma_{allow}$ a fatigue allowance that includes the notch of the seat. For a pinion cut on its shaft, the section at the gear is the root diameter. The journals, which carry almost no moment, are the given shaft diameter. The deflection at the gear, $y = F\,l_A^2 l_B^2/(3EIl)$ for the resultant force on the journal diameter (conservative for a pinion shaft), is reported. Keep it below about $0.01\,m_n$, or raise $K_{H\beta}$ in the rating.

In [ ]:
#| export
# (designation, bore d, outside D, width B, C_r [kN]) — typical catalogue values; confirm with the maker
DEEP_GROOVE_BEARINGS: tuple[tuple[str, float, float, float, float], ...] = (
    ("626", 6, 19, 6, 2.34), ("627", 7, 22, 7, 3.45), ("608", 8, 22, 7, 3.45), ("629", 9, 26, 8, 4.75),
    ("6000", 10, 26, 8, 4.75), ("6200", 10, 30, 9, 5.4), ("6001", 12, 28, 8, 5.4), ("6201", 12, 32, 10, 7.28),
    ("6002", 15, 32, 9, 5.85), ("6202", 15, 35, 11, 8.06), ("6003", 17, 35, 10, 6.37), ("6203", 17, 40, 12, 9.95),
    ("6004", 20, 42, 12, 9.95), ("6204", 20, 47, 14, 13.5), ("6005", 25, 47, 12, 11.9), ("6205", 25, 52, 15, 14.8),
    ("6006", 30, 55, 13, 13.8), ("6206", 30, 62, 16, 20.3), ("6007", 35, 62, 14, 16.8), ("6207", 35, 72, 17, 27.0),
    ("6008", 40, 68, 15, 17.8), ("6208", 40, 80, 18, 32.5),
)


def select_bearing(d_min: float, C_req: float) -> tuple[str, float, float, float, float] | None:
    "Smallest-OD deep groove ball bearing with bore ≥ d_min [mm] and C_r ≥ C_req [N] (None if none listed)."
    ok = [b for b in DEEP_GROOVE_BEARINGS if b[1] >= d_min - 1e-9 and b[4] * 1000 >= C_req]
    return min(ok, key=lambda b: (b[2], b[1], b[3])) if ok else None


def equivalent_load(F_R: float, F_a: float) -> float:
    "Equivalent dynamic load P [N] of a deep groove ball bearing (see above)."
    if F_R > 0 and F_a / F_R <= 0.3:
        return F_R
    return 0.56 * F_R + 1.5 * F_a


def shaft_reactions(F_t: float, F_r: float, F_a: float, r_w: float, span: float, pos: float) -> dict[str, float]:
    "Bearing reactions [N] and bending moment [N·m] of a shaft with the gear at `pos`·span from bearing A."
    lA, lB = pos * span, (1 - pos) * span
    RAt, RBt = F_t * lB / span, F_t * lA / span
    RAr, RBr = (F_r * lB + F_a * r_w) / span, (F_r * lA + F_a * r_w) / span
    M = math.hypot(RAt * lA, max(RAr * lA, RBr * lB)) / 1000
    return {"l_A": lA, "l_B": lB, "F_RA": math.hypot(RAt, RAr), "F_RB": math.hypot(RBt, RBr), "F_aA": F_a, "M": M}

In [ ]:
#| hide
assert select_bearing(15, 5500)[0] == "6002"
assert select_bearing(15, 8000)[0] == "6202"
assert select_bearing(100, 1) is None
assert equivalent_load(1000, 100) == 1000 and math.isclose(equivalent_load(1000, 500), 560 + 750)
_r = shaft_reactions(1000, 400, 0, 20, 60, 0.5)
assert math.isclose(_r["F_RA"], math.hypot(500, 200)) and math.isclose(_r["M"], math.hypot(500, 200) * 30 / 1000)
_r = shaft_reactions(1000, 400, 300, 20, 60, 0.5)           # axial moment adds to both radial reactions
assert _r["F_RA"] > math.hypot(500, 200) and math.isclose(_r["F_RA"], _r["F_RB"])

## Housing Bores

The housing sets the centre distance and the parallelism of the two shafts. The centre distance tolerance is $\pm f_a$, with $f_a$ = IT7/2 of $a_w$, the same value the backlash calculation in `gears.helical.manufacturing` assumes. ISO/TR 10064-3 recommends limits on the inclination and deviation of the shaft axes over the bearing span $L$, from the helix tolerance $F_\beta$ of the gears:

$$f_{\Sigma\beta} = 0.5\,\frac{L}{b}\,F_\beta,\qquad f_{\Sigma\delta} = 2\,f_{\Sigma\beta} .$$

Here $f_{\Sigma\beta}$ is the deviation in the plane of the axes (out-of-plane, which twists the contact), and $f_{\Sigma\delta}$ is the in-plane inclination.

In [ ]:
#| export
@dataclass(frozen=True)
class PairComponents:
    """
    Parts around a helical pair and their checks.

    pinion_hub, gear_hub : dict  joint data (see `size_hub`)
    shafts               : dict  'pinion' / 'gear' → span, reactions, moment, minimum diameter, deflection
    bearings             : dict  'pinion_A', 'pinion_B', 'gear_A', 'gear_B' → loads, C_req, selected bearing
    housing              : dict  centre distance and axis tolerances
    lubrication          : str
    checks               : tuple[Check, ...]
    """
    pinion_hub: dict
    gear_hub: dict
    shafts: dict
    bearings: dict
    housing: dict
    lubrication: str
    checks: tuple[Check, ...]

    @property
    def ok(self) -> bool:
        return all(c.ok for c in self.checks)

    def problems(self) -> list[str]:
        return [str(c) for c in self.checks if not c.ok]


def size_pair_components(pair: HelicalPair, rating: HelicalPairRating, ci: PairComponentInputs,
                         life_h: float, n_in: float, K_A: float = 1.25, grade: int = 6) -> PairComponents:
    "Size hubs, shafts and bearings of the pair (see the sections above)."
    p, g = pair.pinion, pair.gear
    f = pair.forces(rating.T1)
    checks: list[Check] = []
    ph, c1 = size_hub(p, ci.hub_joint1, ci.shaft_d1, rating.T1, f["F_a"], K_A, ci)
    gh, c2 = size_hub(g, ci.hub_joint2, ci.shaft_d2, rating.T2, f["F_a"], K_A, ci)
    checks += c1 + c2
    L_h = ci.bearing_life_h or life_h
    shafts, bearings = {}, {}
    E = 210_000.0
    for key, gear, d, span, r_w, T, n in (("pinion", p, ci.shaft_d1, ci.span1, f["r_w1"], rating.T1, n_in),
                                          ("gear", g, ci.shaft_d2, ci.span2, f["r_w2"], rating.T2, n_in / pair.ratio)):
        sel_prev = None
        B_guess = 8.0
        for _ in range(3):                                    # span depends on the bearing width
            L = span if span is not None else gear.b + B_guess + 6.0
            R = shaft_reactions(f["F_t"], f["F_r"], f["F_a"], r_w, L, ci.gear_position)
            d_min = (32 * K_A * math.hypot(R["M"], math.sqrt(0.75) * T) * 1000 / (math.pi * ci.sigma_shaft_allow)) ** (1 / 3)
            integral = gear is p and ph["joint"] == "integral"
            d_seat = gear.df if integral else d               # section that carries the moment at the gear
            sel = {}
            for side, FR, Fa in (("A", R["F_RA"], R["F_aA"]), ("B", R["F_RB"], 0.0)):
                P = equivalent_load(FR, Fa)
                C_req = P * (60 * abs(n) * L_h / 1e6) ** (1 / 3)
                brg = select_bearing(d, C_req)
                sel[side] = (FR, Fa, P, C_req, brg)
            widths = [s[4][3] for s in sel.values() if s[4] is not None]
            B_new = max(widths) if widths else B_guess
            if span is not None or sel_prev == B_new:
                break
            sel_prev, B_guess = B_new, B_new
        I = math.pi * d ** 4 / 64
        F_res = math.sqrt(f["F_t"] ** 2 + f["F_r"] ** 2)
        y = F_res * R["l_A"] ** 2 * R["l_B"] ** 2 / (3 * E * I * L)
        shafts[key] = {"span": L, "gear_from_A": R["l_A"], "M_max": R["M"], "T": T, "d_min": d_min, "d": d,
                       "section_at_gear": d_seat, "deflection_at_gear": y, "deflection_per_mn": y / gear.mn}
        checks.append(Check(f"{key} shaft section at the gear vs strength minimum"
                            + (" (root diameter)" if integral else ""), d_seat, d_min, "mm", upper=False))
        for side, (FR, Fa, P, C_req, brg) in sel.items():
            name = f"{key}_{side}"
            life = math.inf if brg is None else (brg[4] * 1000 / P) ** 3 * 1e6 / (60 * abs(n)) if P > 0 else math.inf
            bearings[name] = {"role": "located (takes F_a)" if side == "A" else "floating",
                              "F_R": FR, "F_a": Fa, "P": P, "speed_rpm": abs(n), "C_req": C_req,
                              "bearing": (f"{brg[0]} ({brg[1]:g} × {brg[2]:g} × {brg[3]:g}, C_r ≈ {brg[4]:g} kN)"
                                          if brg else "none in the list: use a larger series or angular contact"),
                              "L10h_est": 0.0 if brg is None else life,
                              "bore": brg[1] if brg else d, "D": brg[2] if brg else 2 * d, "width": brg[3] if brg else 8.0}
            checks.append(Check(f"bearing {name} found with C_r ≥ C_req", 0.0 if brg is None else 1.0, 1.0, "", upper=False))
    a_w = pair.a_w
    for side in ("A", "B"):                       # both shafts' bearings sit in the same housing wall
        Dp, Dg = bearings[f"pinion_{side}"]["D"], bearings[f"gear_{side}"]["D"]
        checks.append(Check(f"bearing outer rings {side} clear in the housing wall ((D1 + D2)/2 + {ci.wall_min:g} mm)",
                            (Dp + Dg) / 2 + ci.wall_min, a_w, "mm"))
    Fb = iso1328_tolerances(SimpleNamespace(d=g.d, m=g.mn, b=g.b), grade)["F_betaT"] / 1000  # type: ignore[arg-type]
    L_brg = max(shafts["pinion"]["span"], shafts["gear"]["span"])
    f_sb = 0.5 * L_brg / min(p.b, g.b) * Fb
    housing = {"centre_distance": f"{a_w:.3f} ±{it_grade(a_w, 7) / 2 * 1000:.0f} µm",
               "f_Sigma_beta": f_sb, "f_Sigma_delta": 2 * f_sb, "bearing_span": L_brg,
               "bores": "bearing seats H7 (or K6 for the located bearing under reversing load), coaxial within IT5"}
    return PairComponents(ph, gh, shafts, bearings, housing, lubrication_advice(rating.v), tuple(checks))

In [ ]:
import math
from emachines.gears.helical.layout import HelicalPair
from emachines.gears.helical.rating import rate_helical_pair, HelicalRatingInputs

pair = HelicalPair.build(17, 52, mn=1.5, beta=math.radians(15), b=20)
ci = PairComponentInputs(shaft_d1=15, shaft_d2=25)
r = rate_helical_pair(pair, HelicalRatingInputs(torque_in=20.0, n_in=3000.0, life_h=2000), rims=pair_rims(pair, ci))
c = size_pair_components(pair, r, ci, life_h=2000, n_in=3000.0)
print(c.pinion_hub["joint"], c.gear_hub.get("key"))
for k, v in c.bearings.items():
    print(k, v["bearing"], f"C_req {v['C_req'] / 1000:.2f} kN, L10h ≈ {v['L10h_est']:.3g} h")
print("\n".join(str(x) for x in c.checks))

In [ ]:
#| hide
assert c.pinion_hub["joint"] == "integral"            # Ø15 bore would leave no rim under a Ø23.6 root
assert c.gear_hub["joint"] == "key"
assert all(v["L10h_est"] >= 2000 for v in c.bearings.values())
assert c.bearings["pinion_A"]["F_a"] > 0 and c.bearings["pinion_B"]["F_a"] == 0
assert c.shafts["pinion"]["d_min"] < c.shafts["pinion"]["section_at_gear"] == pair.pinion.df
assert c.ok, c.problems()
_c2 = size_pair_components(pair, r, PairComponentInputs(shaft_d1=15, shaft_d2=25, hub_joint2="press"), 2000, 3000.0)
assert _c2.gear_hub["joint"] == "press" and "F_friction" in _c2.gear_hub

## References

1. ISO 281:2007, *Rolling bearings — Dynamic load ratings and rating life*.
2. ISO/TR 10064-3:1996, *Code of inspection practice — Part 3: Recommendations relative to gear blanks, shaft centre distance and parallelism of axes*.
3. DIN 6885-1:1968, *Parallel keys*; DIN 7190-1:2017, *Interference fits* (via `gears.components`).
4. G. Niemann, H. Winter, B.-R. Höhn, *Maschinenelemente*, Band 1, Springer, 4th ed., 2005 (shaft design).